# PMT bias test: ramp up, one Rigol acquisition, ramp down

1. **Parameters**: edit the cell below.
2. **Connect**: prints the current state of the HV channels and the scope. Nothing changes yet.
3. **Run**: biases the PMTs, takes one acquisition, then biases them back down. Ramp-down is in a
   `finally:` block, so it also happens if the acquisition fails or you interrupt the kernel.
4. **Save / plot**: writes raw Ch1/Ch2 to CSV (no calibration) and plots it.

If anything goes wrong, run the **Emergency HV off** cell at the bottom.

HV voltages are given as **magnitudes**. The sign comes from each channel's polarity setting on the
SHR (channels 0 and 1 are negative, so 100 here means -100 V).

In [ ]:
# ---- HV (ISEG SHR, Ethernet) -------------------------------------------------
HV_RESOURCE = "192.168.0.100"      # IP or full VISA resource
HV_VOLTAGES = {                    # channel -> magnitude in volts (polarity from the SHR)
    0: 100.0,                      # PMT A
    1: 100.0,                      # PMT B
}
HV_CURRENT_LIMIT_A = None          # e.g. 100e-6; None = leave the SHR setting unchanged
HV_RAMP_V_PER_S = None             # e.g. 25; None = leave unchanged (currently 25 V/s)
HV_TOLERANCE_V = 1.0               # |measured - set| before the HV counts as stable
HV_OFF_THRESHOLD_V = 5.0           # |measured| below which a channel counts as down
HV_TIMEOUT_S = 300.0               # max wait for each ramp up / down

# ---- Scope (Rigol MHO954, Ethernet) ------------------------------------------
SCOPE_RESOURCE = "192.168.0.101"   # IP or full VISA resource
CHANNELS = {                       # scope channel -> settings
    1: dict(scale_v_div=0.05, offset_v=0.0, coupling="DC", impedance="OMEG"),  # OMEG = 1 MOhm, FIFTy = 50 Ohm
    2: dict(scale_v_div=0.05, offset_v=0.0, coupling="DC", impedance="OMEG"),
}
TIMEBASE_S_DIV = 100e-9            # horizontal scale, s/div
TIME_OFFSET_S = 0.0                # horizontal offset, s
MEMORY_DEPTH = None                # e.g. "10k"; None = leave as set on the scope

TRIGGER_MODE = "force"             # "force": trigger immediately (always gives a trace)
                                   # "wait":  wait for a real edge on TRIG_SOURCE
                                   # "ad3":   fire one AD3 pulse and trigger on it
TRIG_SOURCE = 2                    # edge-trigger channel
TRIG_LEVEL_V = -0.02               # edge-trigger level, volts
TRIG_SLOPE = "NEG"                 # "POS", "NEG" or "RFAL"
TRIGGER_TIMEOUT_S = 5.0

# ---- AD3 pulse (only used when TRIGGER_MODE == "ad3") -------------------------
AD3_AMPLITUDE_V = 1.0
AD3_WIDTH_S = 100e-9
AD3_PRE_DELAY_S = 100e-9

# ---- Output --------------------------------------------------------------------
OUTPUT_DIR = "runs/bias_test"

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt

from ad3pulse.hardware import IsegHV, RigolScope

hv = IsegHV(HV_RESOURCE).connect()
scope = RigolScope(SCOPE_RESOURCE).connect()
print(f"HV:    {hv.idn}")
print(f"Scope: {scope.idn}")


def hv_report(channels):
    for ch in channels:
        hv.channel = ch
        status = hv.read_status()
        print(f"  HV ch{ch}: pol {hv.read_polarity()}, set {hv.read_set_voltage():8.2f} V, "
              f"meas {hv.read_voltage():8.2f} V, {hv.read_current() * 1e6:7.3f} uA, "
              f"{', '.join(hv.describe_status(status)) or 'off'}")


hv_report(HV_VOLTAGES)

## Run: bias up, acquire, bias down

In [ ]:
def acquire():
    """Configure the Rigol, take one single acquisition and read Ch1/Ch2."""
    for ch, cfg in CHANNELS.items():
        scope.configure_channel(ch, **cfg)
    scope.configure_timebase(TIMEBASE_S_DIV, TIME_OFFSET_S, MEMORY_DEPTH)
    scope.configure_edge_trigger(TRIG_SOURCE, TRIG_LEVEL_V, TRIG_SLOPE)
    status = scope.arm_single(TRIGGER_TIMEOUT_S, allow_early=TRIGGER_MODE != "ad3")
    if TRIGGER_MODE == "force" and status == "WAIT":
        scope.force_trigger()
    elif TRIGGER_MODE == "ad3":
        from ad3pulse.instrument import AnalogDiscovery, PulseSpec
        pulse = PulseSpec(amplitude=AD3_AMPLITUDE_V, width=AD3_WIDTH_S,
                          pre_delay=AD3_PRE_DELAY_S, post_delay=max(1e-6, 4 * AD3_WIDTH_S))
        pulse.fit_to_awg()
        with AnalogDiscovery() as dev:
            dev.configure_pulse(pulse)
            dev.fire(pulse, TRIGGER_TIMEOUT_S)
    try:
        scope.wait_stopped(TRIGGER_TIMEOUT_S)
    except TimeoutError:
        scope._write(":STOP")
        raise
    return scope.capture_channels()


def show(v, _status):
    print(f"\r    {v:9.2f} V", end="", flush=True)


cap = None
hv_readings = {}
try:
    # 1. Bias up. Set VSET before switching on, so a channel never ramps to an old set value.
    for ch, volts in HV_VOLTAGES.items():
        hv.channel = ch
        if HV_CURRENT_LIMIT_A is not None:
            hv.set_current_limit(HV_CURRENT_LIMIT_A)
        if HV_RAMP_V_PER_S is not None:
            hv.set_ramp(HV_RAMP_V_PER_S)
        signed = hv.set_voltage(volts)
        hv.output_on()
        print(f"HV ch{ch}: ramping to {signed:g} V")
    for ch, volts in HV_VOLTAGES.items():
        hv.channel = ch
        print(f"HV ch{ch}: waiting for stable")
        measured = hv.wait_stable(volts, HV_TOLERANCE_V, HV_TIMEOUT_S, on_poll=show)
        current = hv.read_current()
        hv_readings[ch] = (measured, current)
        print(f"\rHV ch{ch}: stable at {measured:.2f} V, {current * 1e6:.3f} uA")

    # 2. One acquisition.
    acquired_at = datetime.now()
    cap = acquire()
    print(f"Scope: {len(cap.time_s)} points @ {cap.sample_rate_Hz:,.0f} S/s")
finally:
    # 3. Bias down, whatever happened above.
    for ch in HV_VOLTAGES:
        hv.channel = ch
        hv.output_off()
        print(f"HV ch{ch}: switched off, ramping down")
    for ch in HV_VOLTAGES:
        hv.channel = ch
        measured = hv.wait_off(HV_OFF_THRESHOLD_V, HV_TIMEOUT_S, on_poll=show)
        print(f"\rHV ch{ch}: down at {measured:.2f} V")

hv_report(HV_VOLTAGES)

## Save and plot
Raw volts from the scope's own preamble, no calibration. HV readings and settings go in the CSV header.

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / f"bias_test_{acquired_at:%Y%m%d_%H%M%S}.csv"

meta = {"acquired_at": acquired_at.isoformat(timespec="seconds"),
        "trigger_mode": TRIGGER_MODE, "trig_source": TRIG_SOURCE,
        "trig_level_V": TRIG_LEVEL_V, "trig_slope": TRIG_SLOPE,
        "timebase_s_div": TIMEBASE_S_DIV, "time_offset_s": TIME_OFFSET_S}
for ch, cfg in CHANNELS.items():
    meta.update({f"ch{ch}_{k}": v for k, v in cfg.items()})
for ch, (v, i) in hv_readings.items():
    meta.update({f"hv{ch}_set_V": HV_VOLTAGES[ch], f"hv{ch}_meas_V": v, f"hv{ch}_meas_I_A": i})
cap.meta = {**meta, **cap.meta}
cap.to_csv(out)
print(f"Saved {out}")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
for ax, ch, trace in ((axes[0], 1, cap.ch1_V), (axes[1], 2, cap.ch2_V)):
    ax.plot(cap.time_s * 1e9, trace * 1e3, lw=0.8, color=f"C{ch - 1}")
    ax.set_ylabel(f"Ch{ch} (mV)")
    ax.grid(alpha=0.3)
hv_text = ", ".join(f"ch{ch} {v:.1f} V" for ch, (v, _) in hv_readings.items())
axes[0].set_title(f"{out.name}  (HV {hv_text})", fontsize=9)
axes[1].set_xlabel("time (ns, 0 = trigger)")
fig.tight_layout()
fig.savefig(out.with_suffix(".png"), dpi=150)
plt.show()

## Close connections

In [ ]:
hv.close()
scope.close()

## Emergency HV off
Switches off **all four** HV channels (with the configured ramp). Works even if the cells above were not run.

In [ ]:
from ad3pulse.hardware import IsegHV

with IsegHV(HV_RESOURCE if "HV_RESOURCE" in dir() else "192.168.0.100") as _hv:
    for _ch in range(4):
        _hv.channel = _ch
        _hv.output_off()
        print(f"HV ch{_ch}: off command sent, measured {_hv.read_voltage():.2f} V")